In [ ]:
import os # para manejar archivos y directorios
import pandas as pd # para manejar datos en formato de tablas
import numpy as np # para manejar datos numéricos y realizar operaciones matemáticas
import matplotlib.pyplot as plt # para crear gráficos y visualizaciones
import seaborn as sns # para crear gráficos estadísticos y visualizaciones más atractivas
from pathlib import Path # para manejar rutas de archivos y directorios de manera más fácil

In [ ]:
# Para definir un objeto path de la librería pathlib
ruta_datos_raw = Path("/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/src/01_download/GoogleStreetView/raw")

In [ ]:
# Establecemos el directorio de trabajo actual
os.chdir("/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/data/raw")

# Especificamos la ruta relativo al archivo CSV y especificamos el delimitador para que no nos de error
df = pd.read_csv("UHogar-csv.tab", sep="\t")
df.head(25) # Mostrar las primeras 5 filas del DataFrame (si la dejamos sin número también muestra las primeras 5 filas)

In [ ]:
#Función para cargar ola y leer el archivo

def cargar_ola(ano: int, cfg: dict) -> pd.DataFrame:
    archivos = cfg["archivos"][ano]
    partes = []
    for zona_archivo, ruta in archivos.items():
        ruta = Path(ruta)
        if ruta.suffix.lower() == ".dta":
            df = pd.read_stata(
                ruta,
                convert_categoricals=False,
            )
        elif ruta.suffix.lower() in [".csv", ".tab"]:
            df = pd.read_csv(
                ruta,
                sep="\t" if ruta.suffix.lower() == ".tab" else ",",
                dtype=str,
                encoding="utf-8",
                on_bad_lines="skip",
            )
        else:
            raise ValueError(f"Extensión no soportada: {ruta.suffix}")
        df["zona_archivo"] = zona_archivo  # marca si la fila viene de "rural" o "urbano"
        partes.append(df)  # acumula este DataFrame en la lista
    df_ola = pd.concat(partes, ignore_index=True)
    df_ola["ola"] = ano
    return df_ola

In [ ]:
# Prueba de la creación de los elementos de un diccionario y sus llaves
cfg = {
    "archivos": {
        2010: {
            "rural": "elca_2010/RHogar-csv.tab",
            "urbano": "elca_2010/UHogar.csv.tab"
        },
        2013: {
            "rural": "elca_2013/RHogar-csv.tab",
            "urbano": "elca_2013/UHogar.csv.tab"
        },
        2016: {
            "rural": "elca_2016/RHogar-csv.tab",
            "urbano": "elca_2016/UHogar.csv.tab"
        },
    }
}

In [ ]:
# para acceder al tipo de cada uno de los elementos del diccionario
type (cfg["archivos"])

In [ ]:
#Para operar la terminal e instalar una librería sin necesidad de hacerlo desde la terminal
import sys
!{sys.executable} -m pip install --upgrade pip

In [ ]:
#Para imprimir la versión del python ejecutable (el que se está utilizando)
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11


In [ ]:
#Función de prueba para unir distintos pdfs desde distintas carpetas

from pathlib import Path
from pypdf import PdfWriter

def unir_pdfs_de_una_carpeta(carpeta, ruta_salida):
    carpeta = Path(carpeta)
    archivos = list(carpeta.iterdir())
    pdfs_encontrados = []
    for archivo in archivos:
        if archivo.suffix.lower() == ".pdf":
            pdfs_encontrados.append(archivo)
    pdfs_encontrados.sort()

    escritor = PdfWriter()
    for pdf in pdfs_encontrados:
        escritor.append(pdf)

    escritor.write(ruta_salida)
    escritor.close()

    return ruta_salida


def unir_pdfs_de_todas_las_carpetas(carpeta_raiz, carpeta_resultados, ruta_salida_final):
    carpeta_raiz = Path(carpeta_raiz)
    carpeta_resultados = Path(carpeta_resultados)
    elementos = list(carpeta_raiz.iterdir())

    subcarpetas = []
    for elemento in elementos:
        if elemento.is_dir():
            subcarpetas.append(elemento)
    subcarpetas.sort()

    pdfs_intermedios = []
    for subcarpeta in subcarpetas:
        nombre_salida = subcarpeta.name + "_unido.pdf"
        ruta_salida_intermedia = carpeta_resultados / nombre_salida
        unir_pdfs_de_una_carpeta(subcarpeta, ruta_salida_intermedia)
        pdfs_intermedios.append(ruta_salida_intermedia)

    escritor_final = PdfWriter()
    for pdf in pdfs_intermedios:
        escritor_final.append(pdf)

    escritor_final.write(ruta_salida_final)
    escritor_final.close()

    return ruta_salida_final


carpeta_raiz = "/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/src/01_download/GoogleStreetView/raw"
carpeta_resultados = "/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/src/01_download/GoogleStreetView/raw/diccionarios_elca"
ruta_salida_final = "/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/src/01_download/GoogleStreetView/raw/diccionarios_elca/diccionarios_elca_unido.pdf"

resultado = unir_pdfs_de_todas_las_carpetas(carpeta_raiz, carpeta_resultados, ruta_salida_final)
print(resultado)

/Users/macbook/Documents/Documentos/tesis_vulnerabilidad/src/01_download/GoogleStreetView/raw/diccionarios_elca/diccionarios_elca_unido.pdf
